In [2]:
"""
run_sweep_B.py — Supplier strategy sweep.

Fixes product=Cola, supplier=LOCAL, model=deepseek-reasoner.
Varies strategy across 6 types.

Usage:
    python run_sweep_B.py
"""

import sys
from pathlib import Path

from openai import OpenAI

from NegoLib.Entities.Agents import LOCAL_SUPPLIER
from NegoLib.Entities.products import COLA

from exp_runner_0412 import (
    batch_random_buyer_exp, build_transitions, save_json,
)

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
CLIENT = OpenAI(
    api_key="sk-81e3e6eae33d4486ad4127e0cb3630a6",         # <-- replace
    base_url="https://api.deepseek.com",
)
MODEL = "deepseek-reasoner"
PRODUCT = COLA
QUANTITY = 60
SUPPLIER = LOCAL_SUPPLIER
N_RUNS = 5
N_ROUNDS = 8
BASE_SEED = 3000

OUTPUT_DIR = Path("results/sweep_B")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

STRATEGIES = [
    "cooperative",
    "anchor_high",
    "gradual_conceder",
    "tit_for_tat_matcher",
    "fairness_responder",
    "walk_away_shutdown",
]

In [ ]:

all_experiments = []

for idx, strategy in enumerate(STRATEGIES, 1):
    print(f"\n[{idx}/{len(STRATEGIES)}]  strategy = {strategy}")

    experiment = batch_random_buyer_exp(
        client=CLIENT,
        model=MODEL,
        product=PRODUCT,
        quantity=QUANTITY,
        supplier=SUPPLIER,
        strategy=strategy,
        n_runs=N_RUNS,
        n_rounds=N_ROUNDS,
        base_seed=BASE_SEED,
        verbose=True,
    )

    save_json(experiment, OUTPUT_DIR / f"B_{strategy}.json")
    all_experiments.append(experiment)

all_trans = []
for exp in all_experiments:
    all_trans.extend(build_transitions(exp))

save_json(
    {"sweep": "B", "n_conditions": len(STRATEGIES),
        "n_transitions": len(all_trans), "transitions": all_trans},
    OUTPUT_DIR / "B_all_transitions.json",
)
print(f"\nSweep B complete.  {len(all_trans)} transitions total.")

# 85 min


[1/6]  strategy = cooperative

--- Run 1/5 (seed=3000) ---
  R1 BUYER : $1.07  — My offer: Cola (330ml): $1.07 per unit, 60 units.
  R1 SELLER: $1.30  ← "Thank you for your offer. While I appreciate you starting the conversation, $1.07 is quite a bit below the established m..."
         [thinking]: "Okay, the merchant's opening offer is $1.07 per unit for 60 units of Cola. My pr..."
  R2 BUYER : $0.67  — How about $0.67 per unit for 60 units of Cola (330ml)?
  R2 SELLER: $1.00  ← "Your latest offer of $0.67 is a significant drop from where we started, and is quite far from the prevailing market rate..."
         [thinking]: "Okay, the merchant just counter-offered $0.67 per unit for 60 units. That's way ..."
  R3 BUYER : $0.64  — I'm willing to pay $0.64 per unit for Cola (330ml), 60 units total.
  R3 SELLER: $0.85  ← "Thank you for your response. However, $0.64 per unit is significantly below the market rate of $1.50 and doesn't reflect..."
         [thinking]: "First, the conversati